In [ ]:
# @title 1. Setup and Configuration
# --- IMPORTANT ---
# 1. Change `zip_file_path` to the location of your input zip file.
# 2. Change `output_directory` to the local folder where you want to save the results.

# --- INPUT ---
zip_file_path = "/content/images.zip"  # <-- CHANGE THIS TO YOUR FILE'S PATH

# --- OUTPUT ---
# The script will create this directory if it doesn't exist.
# Example for Linux/Mac: "/home/user/processed_images"
# Example for Windows: "C:/Users/YourUser/Documents/ProcessedImages"
output_directory = "/content/processed_output" # <-- CHANGE THIS TO YOUR DESIRED OUTPUT FOLDER

# --- Library Imports ---
import os
import zipfile
from PIL import Image
import io
import multiprocessing
from tqdm.notebook import tqdm

# Suppress DecompressionBombError for very large images, use with caution.
Image.MAX_IMAGE_PIXELS = None

print("Configuration and libraries are ready.")
cpu_cores = multiprocessing.cpu_count()
print(f"Using {cpu_cores} CPU cores for processing.")
print(f"Input file: {zip_file_path}")
print(f"Output will be saved to: {output_directory}")

In [ ]:
# @title 2. Worker Function for Multiprocessing (No Changes)
# This function defines the work that each individual process will do.
# It is designed to skip and report files that cause errors.

def process_single_image_worker(image_data):
    """
    Worker function to crop a single image.
    Args:
        image_data (tuple): A tuple containing (filename, image_bytes).
    Returns:
        dict: A dictionary with the original filename and the byte data of the cropped images.
              Returns None if processing fails (e.g., due to corruption).
    """
    filename, image_bytes = image_data
    try:
        # Open image from its byte content
        img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        width, height = img.size

        # --- Crop Right Half ---
        right_half_coords = (width / 2, 0, width, height)
        cropped_right = img.crop(right_half_coords)
        right_buffer = io.BytesIO()
        cropped_right.save(right_buffer, format='JPEG')
        right_bytes = right_buffer.getvalue()

        # --- Crop Top Right Quadrant ---
        top_right_coords = (width / 2, 0, width, height / 2)
        cropped_top_right = img.crop(top_right_coords)
        top_right_buffer = io.BytesIO()
        cropped_top_right.save(top_right_buffer, format='JPEG')
        top_right_bytes = top_right_buffer.getvalue()

        return {
            'filename': os.path.basename(filename),
            'right_half_bytes': right_bytes,
            'top_right_bytes': top_right_bytes
        }
    except Exception as e:
        # This block catches errors like the TIFF read error and prevents a crash.
        print(f"\n⚠️ SKIPPED: Could not process file '{filename}'. Reason: {e}")
        return None

In [ ]:
# @title 3. Main Processing Logic with Local Saving (Updated)
# This cell runs the main logic. It now saves the final zip files
# to the `output_directory` you specified in the first cell.

def process_images_in_parallel(path_to_zip, output_dir):
    if not os.path.exists(path_to_zip):
        print(f"🚨 ERROR: Input file not found at '{path_to_zip}'")
        return

    # --- Create the output directory if it doesn't exist ---
    try:
        os.makedirs(output_dir, exist_ok=True)
        print(f"Output directory is ready at: {os.path.abspath(output_dir)}")
    except Exception as e:
        print(f"🚨 ERROR: Could not create output directory '{output_dir}'. Reason: {e}")
        return

    try:
        # STAGE 1: Read all image data into memory
        image_data_to_process = []
        with zipfile.ZipFile(path_to_zip, 'r') as original_zip:
            all_files = original_zip.infolist()
            print(f"\nReading {len(all_files)} files from zip archive...")
            for item in tqdm(all_files, desc="Reading files"):
                if not item.is_dir() and item.filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif', '.tiff', '.tif')):
                    image_data_to_process.append((item.filename, original_zip.read(item)))

        if not image_data_to_process:
            print("⚠️ Warning: No compatible image files were found in the zip archive.")
            return

        print(f"\nFound {len(image_data_to_process)} images to process.")

        # STAGE 2: Process images in parallel
        results = []
        with multiprocessing.Pool(processes=cpu_cores) as pool:
            with tqdm(total=len(image_data_to_process), desc="Cropping images") as pbar:
                for result in pool.imap_unordered(process_single_image_worker, image_data_to_process):
                    if result:
                        results.append(result)
                    pbar.update(1)

        # STAGE 3: Write results to new zip files in the specified local directory
        print("\nWriting processed images to new zip files...")
        
        # Define the full path for the output files
        right_half_zip_path = os.path.join(output_dir, 'cropped_right_half.zip')
        top_right_quadrant_zip_path = os.path.join(output_dir, 'cropped_top_right_quadrant.zip')

        with zipfile.ZipFile(right_half_zip_path, 'w') as right_zip, \
             zipfile.ZipFile(top_right_quadrant_zip_path, 'w') as top_right_zip:
            for result in tqdm(results, desc="Writing zips"):
                right_zip.writestr(f"right_half_{result['filename']}", result['right_half_bytes'])
                top_right_zip.writestr(f"top_right_{result['filename']}", result['top_right_bytes'])

        print("\n✅ Processing complete!")
        print(f"Successfully saved files to: {os.path.abspath(output_dir)}")
        print(f"  - {right_half_zip_path}")
        print(f"  - {top_right_quadrant_zip_path}")

    except zipfile.BadZipFile:
        print(f"🚨 ERROR: The file '{path_to_zip}' is not a valid zip file.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

# --- Run the main function ---
process_images_in_parallel(zip_file_path, output_directory)